# Fig. 8 — polygon-class fractions over time ($D$ = 0.25, $f_{mut}$ = 0.25)

In [ ]:
from vertex_lite.custom import load_data
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np
import pyprojroot.here as here

In [ ]:
def filter_data(pcts=[0.25], noises=[0.25], local=False, cluster=False, n_simulations=10):
    if cluster:
        rpath = here('results/resultados_cluster')
    else:
        rpath = here('results/resultados_random')

    dfs = []
    for noise in noises:
        for pct in pcts:
            dfs.append(load_data(
                f'{rpath}/noise_{noise}/pct_{pct}/sim_*/cells.parquet',
                columns=['time', 'n_lados', 'neighbor_of_mutant', 'cell_id', 'alive']
            ))

    df = pd.concat(dfs, ignore_index=True)

    if df.empty:
        print("No data found")
        return pd.DataFrame()

    df = df[df['alive'] == 1]

    if local:
        df = df[df['neighbor_of_mutant'] == True]
    else:
        df = df.drop(columns=['neighbor_of_mutant'])

    # Keep every expected simulation/time combination so missing polygon types become zero.
    conditions = df[
        ['time', 'mesh_type', 'pct_mutant']
    ].drop_duplicates()
    simulations = pd.DataFrame({'sim_id': range(n_simulations)})
    simulation_keys = conditions.merge(simulations, how='cross')

    counts = (
        df[(df['n_lados'] > 3) & (df['n_lados'] <= 9)]
        .groupby(['time', 'mesh_type', 'pct_mutant', 'sim_id', 'n_lados'])
        .size()
        .reset_index(name='n_cells')
    )

    polygon_types = pd.DataFrame({'n_lados': range(4, 10)})
    expected = simulation_keys.merge(polygon_types, how='cross')
    df = expected.merge(
        counts,
        on=['time', 'mesh_type', 'pct_mutant', 'sim_id', 'n_lados'],
        how='left'
    )
    df['n_cells'] = df['n_cells'].fillna(0)

    # Fraction within each simulation; absent polygon types contribute zero.
    df['total_cells'] = (
        df.groupby(['time', 'mesh_type', 'pct_mutant', 'sim_id'])['n_cells']
        .transform('sum')
    )
    df = df[df['total_cells'] > 0].copy()  # simulations with no cells in the set are excluded
    df['frac_cells'] = df['n_cells'] / df['total_cells']

    return df

In [ ]:
def summarize(dev, n_simulations=10):
    stats = (
        dev.groupby(["time", "mesh_type", "pct_mutant", "n_lados"])["frac_cells"]
        .agg(["mean", "std", "count"])
        .reset_index()
    )
    if (stats["count"] > n_simulations).any():
        raise ValueError(
            f"Se esperaban {n_simulations} simulaciones por grupo; "
            f"counts encontrados: {stats['count'].unique()}"
        )
    stats["sem"] = stats["std"] / np.sqrt(stats["count"])
    return stats

In [ ]:
def plot(stats, local=False, cluster=False, max_time=5000):
    sns.set_theme(
        style="white",
        context="paper",
        rc={
            "axes.edgecolor": "black",
            "axes.linewidth": 1.0,
            "axes.facecolor": "white",
            "figure.facecolor": "white",
        }
    )

    local_label = "local" if local else "global"
    cluster_label = "cluster" if cluster else "uniform"

    colors_by_lados = {
        4: "#ad7021",
        5: "#e7cf94",
        6: "#8d9695",
        7: "#98d7cd",
        8: "#23867e",
        9: "#003c30",
    }

    for pct in sorted(stats["pct_mutant"].unique()):
        for mesh in sorted(stats["mesh_type"].unique()):
            d0 = stats[
                (stats["pct_mutant"] == pct) &
                (stats["mesh_type"] == mesh)
            ].copy()

            d0 = d0[d0["time"] <= max_time]
            if d0.empty:
                continue

            fig, ax = plt.subplots(figsize=(6.5, 4))

            lados_sorted = sorted(d0["n_lados"].unique())

            for n_lados in lados_sorted:
                d = d0[d0["n_lados"] == n_lados].sort_values("time")
                if d.empty:
                    continue

                x = d["time"].to_numpy()
                y = d["mean"].to_numpy()
                sem = d["sem"].fillna(0).to_numpy()
                color = colors_by_lados[n_lados]

                ax.fill_between(
                    x, y - sem, y + sem,
                    color=color, alpha=0.18, linewidth=0
                )
                ax.plot(
                    x, y,
                    color=color, linewidth=2.25,
                    label=f"{n_lados} lados"
                )

            ax.set_title(
                f"Mutant percentage {pct} | noise {mesh} | {local_label} | {cluster_label}",
                fontsize=14
            )
            ax.set_xlabel("Time step", fontsize=16)
            ax.set_ylabel("Fraction of cells", fontsize=16)

            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)
            ax.spines["bottom"].set_visible(True)
            ax.spines["left"].set_visible(True)
            ax.set_xticks(range(0, max_time + 1, 1000))

            ax.tick_params(
                axis="both", which="both", direction="out",
                length=7, labelsize=18, width=1, bottom=True, left=True
            )
            ax.tick_params(axis='x', rotation=45)
            ax.legend(
                title="Polygon type",
                frameon=False,
                fontsize=11,
                title_fontsize=12
            )
            ax.xaxis.set_ticks_position("bottom")
            ax.yaxis.set_ticks_position("left")
            savedir=(f"{here('results')}/nlados_time_plots/noise_{mesh}/pct_{pct}")
            os.makedirs(savedir, exist_ok=True)
            plt.tight_layout()
            fig.savefig(
                f"{savedir}/polygon_fraction_pct_{pct}_noise_{mesh}_{local_label}_{cluster_label}.svg",
                format="svg"
            )
            plt.close(fig)

In [ ]:
for local in [True, False]:
    for cluster in [True, False]:
        df = filter_data(local=local, cluster=cluster)

        if df.empty:
            continue

        stats = summarize(df)
        plot(stats, local=local, cluster=cluster)